In [1]:
%cd ../.
import os, sys

/home/gtamo/CDD_Vault_API


In [22]:
%load_ext autoreload
%autoreload 2

import os, sys
sys.path.insert(0, os.path.abspath('python'))   # cell [0] cd'd to repo root

import pandas as pd
# custom function
from convert_dataset import convert_to_target_format

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
ori = pd.read_csv('data/MDR1_ori.csv')

In [24]:
col_ori = pd.read_excel('data/mdr1_conversion.xlsx', sheet_name='original') # sheet ori
col_target = pd.read_excel('data/mdr1_conversion.xlsx', sheet_name='target') # sheet target

## Long → wide pivot (no-inhibitor + with-inhibitor side-by-side)

Pipeline:

1. **Strip the `MDR1-MDCK II: ` CDD prefix** from `ori`'s column names — it just records the protocol source and isn't part of the target schema.
2. **Keep only `id_cols + col_ori['Name']`** — drops upstream metadata like `Run Date`, `Run Lab`, etc.
3. **Pivot** using `col_target` as the canonical wide layout: each `col_ori` base column becomes either `<base>` (no-inhibitor row) or `<base> - PgP inhibitor` (with-inhibitor row).

You supply `inhibitor_mask` — a boolean Series aligned to `ori.index` that's `True` for rows measured **with** PgP inhibitor. The mask can come from any column in the original `ori` (even ones not in `col_ori`, since the mask is computed before filtering).

In [25]:
# Canonical home: python/convert_dataset.py — edits there propagate via %autoreload 2.
# Unit tests for this function live in tests/test_convert_dataset.py.


### Example usage

Build the mask from whichever column actually encodes the condition. Two common shapes:

In [26]:
# Build the boolean mask from whichever column in the ORIGINAL `ori`
# (still with the 'MDR1-MDCK II: ' prefix) encodes the inhibitor condition.
# Two common shapes:

# Option A — explicit boolean Series from a known column
inhibitor_mask = ori['MDR1-MDCK II: Cell line'].astype(str).str.contains('inhinitor', case=False, na=False)

# Option B — derive from the comment / study-number field
# inhibitor_mask = ori['MDR1-MDCK II: Study number'].astype(str).str.contains('PgP', na=False)

target = convert_to_target_format(ori, col_ori, col_target, inhibitor_mask)
target.shape, list(target.columns)

WARN: duplicate id_cols in no-inhibitor rows — keeping first
WARN: duplicate id_cols in with-inhibitor rows — keeping first


((90, 28),
 ['Molecule Name',
  'Batch Molecule-Batch ID',
  'Study number',
  'Study number - PgP inhibitor',
  'Date',
  'Date - PgP inhibitor',
  'Cell line',
  'Cell line - PgP inhibitor',
  'Mean Papp A to B',
  'Mean Papp A to B - PgP Inhibitor',
  'Mean Papp B to A',
  'Mean Papp B to A - PgP inhibitor',
  'Efflux ratio',
  'Efflux ratio - PgP inhibitor',
  'Mean %Solution Recovery A to B',
  'Mean %Solution Recovery A to B - PgP inhibitor',
  'Mean %Solution Recovery B to A',
  'Mean %Solution Recovery B to A - PgP inhibitor',
  'Provider Comment',
  'Provider Comment - PgP inhibitor',
  'Serac Comment',
  'Serac Comment - PgP inhibitor',
  'Permeability Class',
  'Permeability Class - PgP inhibitor',
  'Pgp substrate',
  'Pgp substrate - Pgp inhibitor',
  'Provider Name',
  'Provider Name - PgP inhibitor'])

## Visualize the conversion

Eyeball what `convert_to_target_format` did: render the *before* (rows of `ori` for one compound) and the *after* (one row of `target` for the same compound), with color coding:

- **blue** background → no-inhibitor condition (in `ori`) / base columns (in `target`)
- **red** background → with-inhibitor condition (in `ori`) / `- PgP inhibitor` columns (in `target`)
- **gray** background → identifier columns (`Molecule Name`, `Batch Molecule-Batch ID`)

The function is generic — point it at your real `ori` / `target` later by passing them as the first two arguments.

> **Privacy note:** rendered output contains the actual values for the chosen compound. If you commit the notebook, clear outputs first (`jupyter nbconvert --clear-output --inplace …`) to keep chemistry local.

In [27]:
import re
from IPython.display import display, Markdown

from convert_dataset import normalize_ori_columns


def visualize_conversion(
    ori_df: pd.DataFrame,
    target_df: pd.DataFrame,
    compound_names,  # str (single compound) or iterable of str
    col_ori: pd.DataFrame,
    col_target: pd.DataFrame,
    inhibitor_mask: pd.Series,
    id_cols=('Molecule Name', 'Batch Molecule-Batch ID'),
    ori_prefix: str = 'MDR1-MDCK II: ',
    inhibitor_suffix_re: str = r'\s*-\s*Pg[Pp]\s+[Ii]nhibitor\s*$',
    no_inh_color: str = '#cce5ff',   # light blue
    inh_color: str = '#f8d7da',      # light red / pink
    id_color: str = '#f0f0f0',       # light gray
):
    """Side-by-side HTML view of the long→wide conversion for one compound.

    Input: `ori_df` (long-form, with `MDR1-MDCK II: ` prefixes), `target_df`
    (wide-form output of convert_to_target_format), the same `col_ori` /
    `col_target` / `inhibitor_mask` used in the conversion, and the
    `compound_name` to focus on.

    Output: renders two styled DataFrames inline — one ori subset (rows
    colored by inhibitor condition), one target row (columns colored by
    base / inhibitor / id).

    Use after running the conversion to spot-check that values flowed into
    the right cells. Mirrors the in-script pipeline (prefix strip → col_ori
    filter → split → merge) so the visualized ori matches what the function
    actually saw.
    """
    id_cols = list(id_cols)
    # Normalize compound_names to a list for uniform iteration below.
    if isinstance(compound_names, str):
        compound_names = [compound_names]

    for compound_name in compound_names:
        _render_one(
            ori_df, target_df, compound_name, col_ori, col_target,
            inhibitor_mask, id_cols, ori_prefix, inhibitor_suffix_re,
            no_inh_color, inh_color, id_color,
        )


def _render_one(
    ori_df, target_df, compound_name, col_ori, col_target,
    inhibitor_mask, id_cols, ori_prefix, inhibitor_suffix_re,
    no_inh_color, inh_color, id_color,
):
    """Render the before/after view for a single compound (helper for visualize_conversion)."""
    # ---- ori side: normalize column names via the shared helper (prefix
    # strip + unit-suffix rename), then filter to id_cols + col_ori['Name'].
    ori = normalize_ori_columns(ori_df, col_ori, ori_prefix)
    ori_keep = [c for c in list(col_ori['Name']) if c in ori.columns]
    ori = ori[id_cols + ori_keep]

    compound_rows = ori.loc[ori['Molecule Name'] == compound_name]
    if compound_rows.empty:
        display(Markdown(f'**No rows found for compound `{compound_name}` in ori.**'))
        return

    # Align the mask to the filtered subset and order rows: no-inhibitor first
    mask_sub = inhibitor_mask.loc[compound_rows.index]
    sort_idx = mask_sub.sort_values().index
    compound_rows = compound_rows.loc[sort_idx]
    mask_sub = mask_sub.loc[sort_idx]

    def style_ori_rows(row):
        is_inh = bool(mask_sub.loc[row.name])
        color = inh_color if is_inh else no_inh_color
        return [f'background-color: {color}; color: black'] * len(row)

    ori_styled = (
        compound_rows.style
        .apply(style_ori_rows, axis=1)
        .set_caption(
            f'ori (long-form, prefix stripped, filtered to col_ori): '
            f'rows for {compound_name} — blue = no inhibitor, red = with inhibitor'
        )
    )

    # ---- target side: one row for the compound, columns colored by namespace ----
    target_rows = target_df.loc[target_df['Molecule Name'] == compound_name]
    if target_rows.empty:
        display(Markdown(f'**No rows found for compound `{compound_name}` in target.**'))
        return

    suf = re.compile(inhibitor_suffix_re)
    def style_target_cols(s):
        col = s.name
        if col in id_cols:
            color = id_color
        elif suf.search(col):
            color = inh_color
        else:
            color = no_inh_color
        return [f'background-color: {color}; color: black'] * len(s)

    target_styled = (
        target_rows.style
        .apply(style_target_cols, axis=0)
        .set_caption(
            f'target (wide-form): row for {compound_name} — '
            f'blue = base, red = "- PgP inhibitor", gray = id'
        )
    )

    display(Markdown(f'### Conversion view for `{compound_name}`'))
    display(Markdown('**Before** (`ori`, one row per condition):'))
    display(ori_styled)
    display(Markdown('**After** (`target`, one row per compound):'))
    display(target_styled)


# Usage template — uncomment after building inhibitor_mask and target_df on real data
# (see next cell for the full real-data flow).
#
# visualize_conversion(
#     ori, target_df, ori['Molecule Name'].iloc[0],
#     col_ori, col_target, inhibitor_mask,
# )

In [28]:
# Real-data flow: build the mask, run the conversion, visualize the first 2 compounds.
# NOTE: if every inhibitor cell in target_df comes out NaN, your mask is matching
# zero rows — see chat reply for the debug snippet to find the right column.

# Option A — explicit boolean Series from a known column
inhibitor_mask = ori['MDR1-MDCK II: Cell line'].astype(str).str.contains('inhinitor', case=False, na=False)
target_df = convert_to_target_format(ori, col_ori, col_target, inhibitor_mask)

# check whether all col_targets
# Sanity check: target_df has id_cols + col_target['Name'] in order.
assert list(target_df.columns) == ['Molecule Name', 'Batch Molecule-Batch ID'] + list(col_target['Name'])
target_df.head(3)

# Pass a list to render multiple compounds back-to-back.
first_two = target_df['Molecule Name'].iloc[:2].tolist()
visualize_conversion(
    ori, target_df, first_two,
    col_ori, col_target, inhibitor_mask)

WARN: duplicate id_cols in no-inhibitor rows — keeping first
WARN: duplicate id_cols in with-inhibitor rows — keeping first


### Conversion view for `SRB-0002248`

**Before** (`ori`, one row per condition):

,Molecule Name,Batch Molecule-Batch ID,Study number,Date,Cell line,Mean Papp A to B,Mean Papp B to A,Efflux ratio,Mean %Solution Recovery A to B,Mean %Solution Recovery B to A,Provider Comment,Serac Comment,Permeability Class,Pgp substrate,Provider Name
178,SRB-0002248,SRB-0002248-001,426354-20251024-MDR1,nan,MDR1-MDCKⅡ,1.57,10.60,6.79,85.70,93.40,---,nan,Moderate,nan,wuxi
179,SRB-0002248,SRB-0002248-001,426354-20251024-MDR1,nan,MDR1-MDCKⅡ+ PgP inhinitor,2.85,3.70,1.30,98.20,101.00,---,nan,---,nan,wuxi


**After** (`target`, one row per compound):

...

### Conversion view for `SRB-0003109`

**Before** (`ori`, one row per condition):

,Molecule Name,Batch Molecule-Batch ID,Study number,Date,Cell line,Mean Papp A to B,Mean Papp B to A,Efflux ratio,Mean %Solution Recovery A to B,Mean %Solution Recovery B to A,Provider Comment,Serac Comment,Permeability Class,Pgp substrate,Provider Name
176,SRB-0003109,SRB-0003109-001,426354-2025110602-MDR1,nan,MDR1-MDCKⅡ,0.30,15.00,50.70,84.90,96.60,---,nan,Low,nan,wuxi
177,SRB-0003109,SRB-0003109-001,426354-2025110602-MDR1,nan,MDR1-MDCKⅡ+ PgP inhinitor,1.46,3.20,2.20,92.40,99.60,---,nan,---,nan,wuxi


**After** (`target`, one row per compound):

,Molecule Name,Batch Molecule-Batch ID,Study number,Study number - PgP inhibitor,Date,Date - PgP inhibitor,Cell line,Cell line - PgP inhibitor,Mean Papp A to B,Mean Papp A to B - PgP Inhibitor,Mean Papp B to A,Mean Papp B to A - PgP inhibitor,Efflux ratio,Efflux ratio - PgP inhibitor,Mean %Solution Recovery A to B,Mean %Solution Recovery A to B - PgP inhibitor,Mean %Solution Recovery B to A,Mean %Solution Recovery B to A - PgP inhibitor,Provider Comment,Provider Comment - PgP inhibitor,Serac Comment,Serac Comment - PgP inhibitor,Permeability Class,Permeability Class - PgP inhibitor,Pgp substrate,Pgp substrate - Pgp inhibitor,Provider Name,Provider Name - PgP inhibitor
1,SRB-0003109,SRB-0003109-001,426354-2025110602-MDR1,426354-2025110602-MDR1,nan,nan,MDR1-MDCKⅡ,MDR1-MDCKⅡ+ PgP inhinitor,0.300000,1.460000,15.000000,3.200000,50.700000,2.200000,84.900000,92.400000,96.600000,99.600000,---,---,nan,nan,Low,---,nan,nan,wuxi,wuxi


In [29]:
# Check that target_df has exactly the expected columns (id_cols + col_target['Name']).
id_cols = ['Molecule Name', 'Batch Molecule-Batch ID']
expected_cols = id_cols + list(col_target['Name'])
all_match = list(target_df.columns) == expected_cols

# If False, diff so you can see what's off (order or membership).
if not all_match:
    missing = [c for c in expected_cols if c not in target_df.columns]
    extra   = [c for c in target_df.columns if c not in expected_cols]
    print(f'missing_from_target_df ({len(missing)}): {missing}')
    print(f'extra_in_target_df    ({len(extra)}): {extra}')
all_match

True

In [31]:
target_df.to_csv('output/pivoted_mdr1.csv',index=False,sep=',')